# Notebook 02 — LSTM Autoencoder

This notebook:
1. Visualises the encoder-decoder architecture
2. Plots training and validation loss curves
3. Examines reconstruction error distributions (normal vs anomalous)
4. Evaluates the detector and diagnoses its weaknesses

In [ ]:
import sys, json, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, roc_curve

from src.ingestion.features import Normalizer, sliding_windows, time_split
from src.models.lstm_autoencoder import AnomalyDetector

sns.set_theme(style='darkgrid')
DATASET = 'univariate'

## 1. Architecture Overview

```
Input  (batch, seq_len=50, n_features)
  │
  ▼
┌─────────────────────────────────┐
│  LSTM Encoder                   │
│  hidden_size=64, num_layers=2   │
│  dropout=0.2                    │
└──────────────┬──────────────────┘
               │  last hidden state h_n
               ▼
        repeat × seq_len
               │
               ▼
┌─────────────────────────────────┐
│  LSTM Decoder                   │
│  hidden_size=64, num_layers=2   │
└──────────────┬──────────────────┘
               │
               ▼
        Linear(64 → n_features)
               │
               ▼
Output (batch, seq_len, n_features)  — reconstruction

Anomaly score = MSE(input, reconstruction)
```

The model is trained on **normal data only**. At inference time:
- Normal windows → low reconstruction error (the model has seen this pattern)
- Anomalous windows → high reconstruction error (unfamiliar shape)

## 2. Load Training History & Artifacts

In [ ]:
with open(f'../artifacts/{DATASET}_metrics.json') as f:
    metrics = json.load(f)

history = metrics['training_history']

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history['train_loss'], label='Train loss', color='steelblue')
ax.plot(history['val_loss'],   label='Val loss',   color='darkorange')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title(f'LSTM Autoencoder Training — {DATASET}')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Epochs run   : {len(history['train_loss'])}")
print(f"Best val loss: {min(history['val_loss']):.6f}")

## 3. Reconstruction Error Distribution

In [ ]:
df = pd.read_csv(f'../data/sample/{DATASET}.csv', parse_dates=['timestamp'])
_, _, test_df = time_split(df)

feature_cols = ['value'] if DATASET == 'univariate' else ['signal_0', 'signal_1', 'signal_2']

normalizer = Normalizer.load(f'../artifacts/{DATASET}_scaler.pkl')
test_norm  = normalizer.transform(test_df)

detector = AnomalyDetector.load(f'../artifacts/{DATASET}_lstm.pt')
windows, starts = sliding_windows(test_norm, feature_cols, window_size=50, step=1)

# Get labels for each window (label = label of the last point in window)
window_labels = test_df['is_anomaly'].values[starts + 49]
scores = detector.anomaly_score(windows)

fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(scores[window_labels == 0], bins=80, alpha=0.6, color='steelblue', label='Normal')
ax.hist(scores[window_labels == 1], bins=80, alpha=0.6, color='red', label='Anomaly')
ax.axvline(detector.threshold, color='black', ls='--', lw=1.5, label=f'Threshold ({detector.threshold:.5f})')
ax.set_xlabel('Reconstruction Error (MSE)')
ax.set_ylabel('Window count')
ax.set_title('LSTM Reconstruction Error — Normal vs Anomalous Windows')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Test Set Evaluation

In [ ]:
_, window_flags = detector.predict(windows)

# Map window flags back to rows
row_flags = np.zeros(len(test_df), dtype=int)
for i, start in enumerate(starts):
    if window_flags[i]:
        row_flags[start:start+50] = 1

test_labels = test_df['is_anomaly'].values
aligned = min(len(test_labels), len(row_flags))

print(f"Precision : {precision_score(test_labels[:aligned], row_flags[:aligned], zero_division=0):.4f}")
print(f"Recall    : {recall_score(test_labels[:aligned], row_flags[:aligned], zero_division=0):.4f}")
print(f"F1        : {f1_score(test_labels[:aligned], row_flags[:aligned], zero_division=0):.4f}")

# ROC curve using window-level scores & labels
fpr, tpr, _ = roc_curve(window_labels, scores)
auc = roc_auc_score(window_labels, scores)
print(f"ROC-AUC   : {auc:.4f}  (window-level)")

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='darkorange', label=f'AUC = {auc:.4f}')
ax.plot([0,1],[0,1],'--', color='grey')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curve — LSTM Autoencoder')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Threshold Sensitivity

The LSTM's threshold was set at the 95th percentile of normal validation reconstruction errors.
Here we see how F1 changes with different percentile choices.

In [ ]:
# Compute F1 at different threshold percentiles using window-level scores
percentiles = range(50, 100, 2)
normal_scores = scores[window_labels == 0]

f1s, precs, recs = [], [], []
for pct in percentiles:
    thresh = np.percentile(normal_scores, pct)
    preds = (scores > thresh).astype(int)
    f1s.append(f1_score(window_labels, preds, zero_division=0))
    precs.append(precision_score(window_labels, preds, zero_division=0))
    recs.append(recall_score(window_labels, preds, zero_division=0))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(list(percentiles), f1s,   label='F1',        color='steelblue')
ax.plot(list(percentiles), precs, label='Precision',  color='darkorange')
ax.plot(list(percentiles), recs,  label='Recall',     color='green')
ax.axvline(95, color='black', ls='--', lw=1, label='Current threshold (95th pct)')
ax.set_xlabel('Threshold percentile (of normal val scores)')
ax.set_ylabel('Score')
ax.set_title('LSTM Metrics vs Threshold Choice')
ax.legend()
plt.tight_layout()
plt.show()